# RISSK — scheduled pipeline runner (JupyterHub Notebook Jobs)

Each **configuration is a Kedro environment** (`conf/<config>/globals.yml`). This notebook runs
the pipeline **in-process** via `KedroSession` (plain Python, no subprocess): set `ENV` (which
configuration) and `PIPELINE` (which stage) below. There is no driver — the pipeline, storage,
and questionnaire all come from Kedro config.

**A configuration** (`conf/<config>/globals.yml`) sets:
- `survey` + a single `questionnaire` (`name`, `VERSION`, optional `filter_var`) — one questionnaire per env;
- three storage roots, whose *values* pick the storage mode:
  - `input_root` — where the export zips live: a local path, or `s3://<bucket>`.
  - `work_root` — always-local staging (zips are fetched + unzipped here; unzip is local-only).
  - `output_root` — where stages 20→40 land: a local path, or `s3://<bucket>` (written via `s3fs`, no aws CLI).

  → `local` / `s3in` / `s3out` / `s3` are just the four combinations of local vs `s3://` roots.

**`PIPELINE`** selects which stage to run: `"__default__"` (all of data_ingestion → feature_creation → rissk_scoring), or a single one of those.

Outputs are keyed by `<survey>`, so a survey folder holds **one** questionnaire's results.
**Several questionnaires = several envs**, each pointing at its own survey folder; set `ENV` to a list below to run them in one job.

**Prerequisites**

- Environment installed from the repo root: `conda env create -f environment.yml`
  (or `uv sync`). The single `rissk` package is installed editable.
- Kernel registered so Notebook Jobs can run on it (the kernel keeps the name `rissk_kedro`):
  `python -m ipykernel install --user --name rissk_kedro`
- For S3 roots: AWS credentials in the environment (standard chain — env vars or `~/.aws`; `s3fs` uses them).

The cell below is tagged `parameters`, so to schedule a different run you only override `ENV`
(and optionally `PIPELINE`) in the Notebook Jobs *Parameters* form (e.g. `ENV = "grdslchbs_test"`)
— one job per configuration, same notebook, no code changes.

In [1]:
# Which configuration(s) to run — a Kedro env name under conf/<ENV>/, or a list of them.
# (Available envs are the conf/<name>/ folders, e.g. grdslchbs_test, s3in, s3out, s3.) 
# You can create your own env by copying one of the existing ones and modifying it.
ENV = "svgslchbs"

# Which pipeline to run: "__default__" (all stages) or one of
# "data_ingestion" / "feature_creation" / "rissk_scoring".
PIPELINE = "__default__"

In [2]:
import os
import sys
from pathlib import Path


def _find_project_root() -> Path:
    """Locate the project root, resiliently for scheduled Notebook Jobs.

    Prefer the installed (editable) package location — correct on any machine. If the
    scheduled job runs in an environment where the editable install didn't take (e.g.
    `conda env update` ran from the wrong dir, so `-e .` never installed rissk), fall
    back to an explicit path and put src/ on sys.path so `import rissk` still works.
    Set RISSK_PROJECT_ROOT to override the fallback.
    """
    try:
        import rissk
        return Path(rissk.__file__).resolve().parents[2]
    except ModuleNotFoundError:
        root = Path(os.environ.get("RISSK_PROJECT_ROOT", Path.home() / "rissk")).resolve()
        sys.path.insert(0, str(root / "src"))
        return root


PROJECT_ROOT = _find_project_root()

# Anchor the working directory to the project root. Kedro resolves relative dataset paths
# (e.g. work_root: "data") against the project root, but stage_zips writes staged zips
# relative to the CWD. A JupyterHub Notebook Job runs this notebook from a COPIED job dir
# (/jobs/<id>/), so without this the two disagree and ingestion fails with
# "No partitions found in <project_root>/data/.../10_RAW".
os.chdir(PROJECT_ROOT)

# Diagnostics — compare these between an interactive run and a scheduled job. A different
# `interpreter` is the tell-tale that the scheduler is not using the rissk_kedro env.
print("interpreter :", sys.executable, flush=True)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

import rissk  # noqa: F401  (importable now, via the editable install or the src/ fallback)
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

bootstrap_project(PROJECT_ROOT)   # load project settings + pipelines once (env-independent)

envs = ENV if isinstance(ENV, (list, tuple)) else [ENV]

interpreter : /home/jupyter-thescheduler/.conda/envs/rissk_kedro/bin/python
PROJECT_ROOT: /home/jupyter-thescheduler/rissk


[08/20/26 08:10:12] INFO     Using 'conf/logging.yml' as logging configuration. You can change this ]8;id=12567097;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=12567098;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/framework/project/__init__.py#269\269]8;;\
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

In [3]:
from rissk.run import run_survey

# run_survey iterates a survey env's questionnaires (conf/<env>/questionnaires/*.yml),
# running the full pipeline once per questionnaire (per-<qnr> output subfolders), then
# unions their microdata into the survey-level 30_PROCESSED/microdata.parquet. A legacy
# single-questionnaire env (questionnaire in globals.yml, no questionnaires/ folder) runs
# once, unchanged, with no combine step. Failures are isolated per questionnaire.
results = {}
for env in envs:
    outcomes = run_survey(env, project_root=PROJECT_ROOT, pipeline=PIPELINE)
    for label, outcome in outcomes.items():
        results[f"{env}:{label}"] = outcome

ok = sum(r == "OK" for r in results.values())
print(f"\nSummary: {ok}/{len(results)} run(s) succeeded.", flush=True)
failed = [k for k, r in results.items() if r != "OK"]
if failed:
    raise RuntimeError(f"kedro run failed for: {failed}")

                    INFO     NumExpr defaulting to 4 threads.                                          ]8;id=12567105;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/numexpr/utils.py\utils.py]8;;\:]8;id=12567106;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/numexpr/utils.py#164\164]8;;\

=== run --env svgslchbs --pipeline __default__ [svg_slchbs2026] ===


[08/20/26 08:10:13] INFO     Kedro project rissk                                                     ]8;id=12567113;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/framework/session/session.py\session.py]8;;\:]8;id=12567114;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/framework/session/session.py#335\335]8;;\

[08/20/26 08:10:16] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=12567121;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=12567122;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro_telemetry/plugin.py#273\273]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Using synchronous mode for loading and saving data. Use the    ]8;id=12567129;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/runner/sequential_runner.py\sequential_runner.py]8;;\:]8;id=12567130;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/runner/sequential_runner.py#59\59]8;;\
                             --async flag for potential performance gains.                                         
                             https://docs.kedro.org/en/stable/build/run_a_pipeline/#load-an                        
                             d-save-asynchronously                                                                 

...

                    INFO     combine_microdata: adding partition 'svg_slchbs2026'                        ]8;id=12568014;file:///home/jupyter-thescheduler/rissk/src/rissk/run.py\run.py]8;;\:]8;id=12568015;file:///home/jupyter-thescheduler/rissk/src/rissk/run.py#132\132]8;;\

                    INFO     combine_microdata: unioned 1 partitions -> 0 rows                           ]8;id=12568021;file:///home/jupyter-thescheduler/rissk/src/rissk/run.py\run.py]8;;\:]8;id=12568022;file:///home/jupyter-thescheduler/rissk/src/rissk/run.py#142\142]8;;\

                    INFO     Saving data to microdata_combined (ParquetDataset)...             ]8;id=12568027;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=12568028;file:///home/jupyter-thescheduler/.conda/envs/rissk_kedro/lib/python3.13/site-packages/kedro/io/data_catalog.py#1008\1008]8;;\

--- combine: OK ---

Summary: 1/2 run(s) succeeded.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:18                                                                                   │
│                                                                                                  │
│   15 print(f"\nSummary: {ok}/{len(results)} run(s) succeeded.", flush=True)                      │
│   16 failed = [k for k, r in results.items() if r != "OK"]                                       │
│   17 if failed:                                                                                  │
│ ❱ 18 │   raise RuntimeError(f"kedro run failed for: {failed}")                                   │
│   19                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
RuntimeError: kedro run failed for: ['svgslchbs:svg_slchbs2026']

### Where results land

Outputs are written under `output_root`, keyed by survey:

```
<output_root>/<survey>/latest/
    20_INTERIM/   ...
    30_PROCESSED/ ...
    35_SCORES/    item_scores.parquet, responsible_scores.csv, unit_rissk_scores.csv
```

- local roots → on disk; `s3://<bucket>` roots → written natively via `s3fs` (no upload step).
- input zips are always staged + unzipped under the local `work_root` first (unzip is local-only).
- a survey folder holds **one** questionnaire's results — for several questionnaires, use several envs pointing at separate survey folders.

The final scores file is `35_SCORES/unit_rissk_scores.csv` — point downstream consumers there.

For a **multi-questionnaire** survey env (a `conf/<env>/questionnaires/` folder with one yaml per questionnaire), each questionnaire's stage outputs land under a `<qnr>/` subfolder (e.g. `30_PROCESSED/<qnr>/microdata.parquet`), and their microdata is additionally unioned into the survey-level `30_PROCESSED/microdata.parquet`. A legacy single-questionnaire env keeps the flat layout.

In [ ]:
# Post-run cleanup — remove this run's staged input ZIPs from the local work_root.
#
# Why: stage_input_zips_node (data_ingestion) decides whether to re-fetch a zip from
# input_root using a SIZE-ONLY check (rissk.utils.storage.stage_zips). So a changed
# zip in S3 whose byte size is unchanged would be skipped ("Already staged, size match")
# and the OLD staged copy reprocessed. Deleting the staged zip forces a fresh fetch on
# the next run, and frees disk on JupyterHub. Only the zips are removed — the extracted
# folders are left in place (they are rmtree'd + rebuilt at the start of every run anyway).
#
# Guard: skip when work_root IS the input source (input_root == work_root, or an s3://
# work_root). In those envs 10_RAW holds the REAL input zips, not staged copies — never
# delete them. Safe to purge only when input_root and work_root differ (e.g. s3 in / local work).
#
# NOTE: the run cell above raises on failure, so this cell only runs after a fully
# successful run. That's usually what you want (fresh inputs after a good run); move it
# before that raise if you also want cleanup after a failed run.
import yaml
from rissk.run import load_questionnaire_configs

for env in envs:
    with open(PROJECT_ROOT / "conf" / env / "globals.yml") as fh:
        g = yaml.safe_load(fh) or {}
    work_root, input_root, survey = g.get("work_root"), g.get("input_root"), g.get("survey")

    if not work_root or not survey:
        print(f"[{env}] work_root/survey not set in globals — skipping cleanup", flush=True)
        continue
    if str(work_root).startswith("s3://") or work_root == input_root:
        print(f"[{env}] input_root == work_root (staged in place) — skipping to protect real inputs", flush=True)
        continue

    # Questionnaire name(s) this env processes — multi (conf/<env>/questionnaires/*.yml) or single (globals).
    qnrs = load_questionnaire_configs(env, PROJECT_ROOT)
    names = [q["name"] for q in qnrs] if qnrs else (
        [g["questionnaire"]["name"]] if (g.get("questionnaire") or {}).get("name") else [])

    raw = PROJECT_ROOT / work_root / survey / "latest" / "10_RAW"
    removed = []
    if raw.exists():
        for f in raw.iterdir():
            # Match stage_zips' <name>_*.zip scope so shared, survey-level 10_RAW keeps
            # other questionnaires' zips untouched.
            if f.is_file() and f.suffix.lower() == ".zip" and any(f.name.startswith(f"{n}_") for n in names):
                f.unlink()
                removed.append(f.name)
    print(f"[{env}] removed {len(removed)} staged zip(s): {sorted(removed)}" if removed
          else f"[{env}] no matching staged zips under {raw}", flush=True)
